# CGFT Experiment Template

A minimal 3-step pipeline to train an LLM using reinforcement learning:

1. **Create dataset** — Provide prompts (and optiona\lly ground truth) for training and evaluation
2. **Create environment** — Define your RL environment: system prompt, reward function, and (optionally) tools
3. **Run training job** — Upload and launch the experiment on the CGFT platform

## Step 0: Setup

Install dependencies and configure API credentials.

In [ ]:
# For Google Colab - clone repo and install
!pip install git+https://github.com/cgftinc/cgft.git

In [ ]:
# Configuration
# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = ""
BASE_URL = "https://app.cgft.io"

# Optional: name for your training experiment
EXPERIMENT_NAME = "my_first_experiment"
EXPERIMENT_PREFIX = "first-exp"


## Step 1: Create Your Dataset

Provide a list of examples for training. Each example needs at minimum a `"prompt"` — what the model sees. You can optionally include `"ground_truth"` or any other fields your reward function needs.

In [ ]:
dataset = [
    #example: {"prompt": "What is the largest planet in our solar system?", "ground_truth": "Jupiter"}
]

# ── Train/eval split ───
import random
random.seed(42)
TRAIN_EVAL_SPLIT = 0.7

random.shuffle(dataset)
split = int(len(dataset) * TRAIN_EVAL_SPLIT)
train_data, eval_data = dataset[:split], dataset[split:]

## Step 2: Create Environment

An environment defines **how the model is evaluated** during training. You need three things:

1. **System prompt** — Instructions sent to the model before each rollout. Tell it what task to solve and how to format its answer.
2. **Reward function** — Scores the model's response. This is the training signal. You can score however you like: execute code and check correctness, apply a LLM judge, compare against a reference, etc.
3. **Tools** *(optional)* — If your task requires the model to call tools (e.g., search, code execution), define them here. For purely cognitive tasks, leave tools empty.

Customize the `YourEnv` class below for your task.

In [ ]:
import re
from typing import Any

from benchmax.envs.base_env import BaseEnv
from benchmax.envs.types import StandardizedExample, ToolDefinition

# ── System prompt ──
# This is sent to the model at the start of every rollout.
# Tell the model what task it's doing and how to format its answer.
SYSTEM_PROMPT = """
You are solving [TASK DESCRIPTION].
Put your final answer inside <answer></answer> tags.
"""

# ── Environment class ───
class YourEnv(BaseEnv):
    """Your custom RL environment. Rename this class to match your task."""

    system_prompt: str = SYSTEM_PROMPT

    def __init__(self, **kwargs):
        pass  # Add any env-level setup here

    # Maps each row of your dataset into the standardized format the trainer expects.
    # "prompt" becomes what the model sees. "ground_truth" is passed to compute_reward
    # (optional — set to None if your reward doesn't need it).
    @classmethod
    def dataset_preprocess(cls, example: Any, **kwargs) -> StandardizedExample:
        return StandardizedExample(
            prompt=example.get("prompt"),
            ground_truth=example.get("answer"),  # Optional — set to None if not needed
            init_rollout_args={},
        )

    # Define any tools your model can use  (e.g. a search tool).
    # If you don't need tools, just return an empty list.
    async def list_tools(self) -> list[ToolDefinition]:
        return []

    # No-op when there are no tools
    async def run_tool(self, rollout_id: str, tool_name: str, **tool_args) -> Any:
        return ""

    # Define your reward function here. You may have multiple reward components,
    # each one should be a separate entry in the return dict
    async def compute_reward(
        self, rollout_id: str, completion: str, ground_truth: Any, **kwargs: Any
    ) -> dict[str, float]:

        rewards = {}
        rewards["correctness"] = 0.0
        return rewards

## Step 3: Train Model

Upload the dataset and bundle the environment for training.

In [ ]:
import cgft
from cgft.trainer.pipeline import train

experiment_id = train(
    env_class=YourEnv,
    env_args={},
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix=EXPERIMENT_PREFIX,
    api_key=API_KEY,
    base_url=BASE_URL,
    experiment_name=EXPERIMENT_NAME,
)

print(f"Experiment ID: {experiment_id}")

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:
- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse